In [ ]:
import os
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import statsmodels.formula.api as smf

In [ ]:
os.chdir('..')

In [ ]:
MODEL_DATA_PATH = "data/modeling/model_data.geojson"
BOUNDARIES_PATH = "data/lots/city_boundaries.geojson"
WALK_INDEX_PATH = "data/filtered_block_groups/walk_index.geojson"

In [ ]:
walk_index = gpd.read_file(WALK_INDEX_PATH)

In [ ]:
model_data = gpd.read_file(MODEL_DATA_PATH)

In [ ]:
boundaries = gpd.read_file(BOUNDARIES_PATH)
boundaries.to_crs(epsg=5070, inplace=True)
boundaries["boundary_area"] = boundaries.geometry.area

In [ ]:
walk_index_cities = gpd.overlay(walk_index, boundaries, how='intersection')

In [ ]:
walk_index_cities["intersection_area"] = walk_index_cities.geometry.area
walk_index_cities["overlap_pct"] = walk_index_cities["intersection_area"] / walk_index_cities["boundary_area"]
walk_index_cities["weighted_walk_score"] = walk_index_cities["NatWalkInd"] * walk_index_cities["overlap_pct"]

city_walk_index = walk_index_cities.groupby("id_1").agg({
    "weighted_walk_score": "sum",
    "overlap_pct": "sum"
}).reset_index()

In [ ]:
model_data = model_data.merge(
    city_walk_index[["id_1", "weighted_walk_score"]], left_on="city", right_on="id_1", how="left"
).drop(["id_1"], axis=1)

In [ ]:
model_data.head()

In [ ]:
plt.figure(figsize=(8, 6))
sns.histplot(model_data["weighted_walk_score"], bins=25, kde=True)
plt.title('Walk Score Distribution')

In [ ]:
model_data["transformed_weighted_walk_score"] = np.log(model_data["weighted_walk_score"].max() + 1 - model_data["weighted_walk_score"])

In [ ]:
model_data["logit_car_trip_share"] = np.log(model_data["car_trip_share"] / (1 - model_data["car_trip_share"]))

In [ ]:
sns.regplot(data=model_data, x="transformed_weighted_walk_score", y="logit_car_trip_share")
plt.show()

In [ ]:
model_lots = smf.ols(formula = 'logit_car_trip_share ~ transformed_weighted_walk_score', data = model_data).fit()
print(model_lots.summary())